In [ ]:
import sys
sys.path.append('./Textual-Anomaly-Detection-Framework/Anomaly Detection Framework')

from Data_Preparation.Embedding import embedding_encoder
from Data_Preparation.Tac import tac
from Data_Preparation import utils
from Modelisation.FlowMatching import flow_matching
from Modelisation.Baselines.OCSVM import ocsvm
from Modelisation.Baselines.CVDD.utils import build_vocab, cvdd_model_pipeline
import Modelisation.evaluation as ev
from Modelisation.Baselines.CVDD.networks import cvdd_Net
from Modelisation.Baselines.RSRAE.model import CAE
from Modelisation.Baselines.TCCM.model import *
from utils import save_results, create_tables, save_hyperparameters, load_hyperparams

import torch
from torch import Tensor
from torch.utils.data import TensorDataset, DataLoader
import optuna
import torch
from torch import nn, Tensor
import numpy as np
from transformers import AutoTokenizer
from datasets import Dataset, concatenate_datasets
import time
import tensorflow as tf
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from pyod.models.auto_encoder import AutoEncoder
import os

import warnings
warnings.filterwarnings('ignore')

BATCH_SIZE = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
train_20ng_, test_20ng_ = utils.import_dataset(name="20newsgroups", batch_size=BATCH_SIZE)
# train_reuters_, test_reuters_ = utils.import_dataset(name="reuters", batch_size=BATCH_SIZE)
# train_wos = utils.import_dataset(name="WOS", batch_size=BATCH_SIZE)
# train_dbpedia14_, test_dbpedia14_ = utils.import_dataset(name="dbpedia14", batch_size=BATCH_SIZE)
# train_agnews_, test_agnews_ = utils.import_dataset(name="agnews", batch_size=BATCH_SIZE)

In [ ]:
# train_agnews_ = utils.preprocess(train_agnews_.dataset)
# test_agnews_ = utils.preprocess(test_agnews_.dataset)

# train_reuters_ = utils.preprocess(train_reuters_.dataset)
# test_reuters_ = utils.preprocess(test_reuters_.dataset)

train_20ng_ = utils.preprocess(train_20ng_.dataset)
test_20ng_ = utils.preprocess(test_20ng_.dataset)

# train_dbpedia14_ = utils.preprocess(train_dbpedia14_.dataset)
# test_dbpedia14_ = utils.preprocess(test_dbpedia14_.dataset)

In [ ]:
# model_name = 'all-MiniLM-L6-v2'
model_name = 'all-distilroberta-v1'
sentencebertEncoder = embedding_encoder.EmbeddingEncoder(model_name, 'sentencebert', device)

In [ ]:
def train_test_val_split(train, test, inlier_topic, dataset_name, type_tac, anomaly_rate, verbose=False):
    
    train_inlier, train_anomaly = tac.textual_anomaly_contamination(train, dataset_name, inlier_topic, type_tac, anomaly_rate, True)

    n_inliers_val = int(0.1 * len(train_inlier))
    inlier_indices = np.random.choice(len(train_inlier), n_inliers_val, replace=False)
    val_inlier_dataset = train_inlier.select(inlier_indices)

    train_inlier = train_inlier.select([i for i in range(len(train_inlier)) if i not in inlier_indices])
    
    n_anomalies_val = int(n_inliers_val / 0.9 * 0.1)
    anomaly_indices = np.random.choice(len(train_anomaly), n_anomalies_val, replace=False)
    val_anomaly_dataset = train_anomaly.select(anomaly_indices)

    val_ = concatenate_datasets([val_inlier_dataset, val_anomaly_dataset]).shuffle(seed=42)
    
    if verbose:
        print("TRAINSET")
        print(train_inlier)
        print(train_anomaly)
    
    if verbose:
        print("\nVALSET")
        print(val_)
        print()

    test_ = tac.textual_anomaly_contamination(test, dataset_name, inlier_topic, type_tac, anomaly_rate, False)
    # print(test_.filter(lambda x : x['anomaly_class']== 0))
    # print(test_.filter(lambda x : x['anomaly_class']== 1))
    if verbose:
        print("TESTSET")
        print(test_)

    return train_inlier, train_anomaly, val_, test_

def get_embeddings(sentencebertEncoder, train_reuters_, test_reuters_, inlier_topic, dataset_name, type_tac, anomaly_rate, hm='all', text_column='text'):

    train_inlier_reuters, train_anomaly_reuters, val_reuters, test_reuters = train_test_val_split(train_reuters_, test_reuters_, inlier_topic, dataset_name, type_tac, anomaly_rate, False)
    
    if hm == 'cvdd':
        return train_inlier_reuters, train_anomaly_reuters, val_reuters, test_reuters

    if hm == 'all':
        train_inlier_reuters = sentencebertEncoder.forward(train_inlier_reuters, text_column)
        test_reuters = sentencebertEncoder.forward(test_reuters, text_column)
        X_inlier = Tensor(train_inlier_reuters['sbert_embeddings']).to(device)
        X_test =  Tensor(test_reuters['sbert_embeddings']).to(device)
        y_test = np.array(test_reuters['anomaly_class'])

        return train_inlier_reuters, test_reuters, X_inlier, X_test, y_test
    
    elif hm == 'train':
        train_inlier_reuters = sentencebertEncoder.forward(train_inlier_reuters, text_column)
        X_inlier = Tensor(train_inlier_reuters['sbert_embeddings']).to(device)
        
        
        return train_inlier_reuters, _, X_inlier, _, _
    
    elif hm == 'test':
        test_reuters = sentencebertEncoder.forward(test_reuters, text_column)
        X_test =  Tensor(test_reuters['sbert_embeddings']).to(device)
        y_test = np.array(test_reuters['anomaly_class'])
        
        return _, test_reuters, _, X_test, y_test 


### Saving data

In [ ]:
save_dir = "/home/2017025/ayouce01/Textual-Anomaly-Detection-Framework/Anomaly Detection Framework/Data"

dataset_name = '20newsgroups'
type_tac = 'ruff' 
anomaly_rate = 0.1

inlier_topics = ['computer', 'recreation', 'science', 'miscellaneous', 'politics', 'religion']

for inlier_topic in inlier_topics:
    
    save_path_temp = os.path.join(save_dir, f"{dataset_name}/{inlier_topic}")
    os.makedirs(save_path_temp, exist_ok=True)
                                 
    save_path = os.path.join(save_path_temp, f"ds_train_{inlier_topic}.pt")
    
    if os.path.exists(save_path):
        print(f"{save_path} already done")
    else:
        _, _, X_inlier, _, _ = get_embeddings(sentencebertEncoder, train_20ng_, test_20ng_, inlier_topic, dataset_name, type_tac, anomaly_rate)    
        print(f"data saved at : {save_path}")
        torch.save({"X_inlier": X_inlier}, save_path)

In [ ]:
save_dir = "/home/2017025/ayouce01/Textual-Anomaly-Detection-Framework/Anomaly Detection Framework/Data"

dataset_name = '20newsgroups'
type_tac = 'ruff' 
anomaly_rate = 0.1

inlier_topics = ['computer', 'recreation', 'science', 'miscellaneous', 'politics', 'religion']
nb_runs = 10

for run in range(nb_runs):

    for inlier_topic in inlier_topics:

        save_path_temp = os.path.join(save_dir, f"{dataset_name}/{inlier_topic}/run{run+1}")
        os.makedirs(save_path_temp, exist_ok=True)

        save_path = os.path.join(save_path_temp, f"ds_test_{inlier_topic}_run{run+1}.pt")
        
        if os.path.exists(save_path):
            print(f"{save_path} already done")
        else:
            _, _, _, X_test, y_test = get_embeddings(sentencebertEncoder, train_20ng_, test_20ng_, inlier_topic, dataset_name, type_tac, anomaly_rate)
            print(f"data saved at : {save_path}")
            torch.save({"X_test": X_test, "y_test": y_test}, save_path)

In [ ]:
save_dir = "/home/2017025/ayouce01/Textual-Anomaly-Detection-Framework/Anomaly Detection Framework/Data"

dataset_name = 'reuters'
type_tac = 'ruff' 
anomaly_rate = 0.1

inlier_topics = ['acq', 'earn', 'trade', 'crude', 'interest', 'money-fx', 'ship']

for inlier_topic in inlier_topics:
    
    save_path_temp = os.path.join(save_dir, f"{dataset_name}/{inlier_topic}")
    os.makedirs(save_path_temp, exist_ok=True)
                                 
    save_path = os.path.join(save_path_temp, f"ds_train_{inlier_topic}.pt")
    
    if os.path.exists(save_path):
        print(f"{save_path} already done")
    else:
        _, _, X_inlier, _, _ = get_embeddings(sentencebertEncoder, train_reuters_, test_reuters_, inlier_topic, dataset_name, type_tac, anomaly_rate)
        print(f"data saved at : {save_path}")
        torch.save({"X_inlier": X_inlier}, save_path)

In [ ]:
save_dir = "/home/2017025/ayouce01/Textual-Anomaly-Detection-Framework/Anomaly Detection Framework/Data"

dataset_name = 'reuters'
type_tac = 'ruff' 
anomaly_rate = 0.1

inlier_topics = ['acq', 'earn', 'trade', 'crude', 'interest', 'money-fx', 'ship']
nb_runs = 10

for run in range(nb_runs):

    for inlier_topic in inlier_topics:

        save_path_temp = os.path.join(save_dir, f"{dataset_name}/{inlier_topic}/run{run+1}")
        os.makedirs(save_path_temp, exist_ok=True)

        save_path = os.path.join(save_path_temp, f"ds_test_{inlier_topic}_run{run+1}.pt")
        
        if os.path.exists(save_path):
            print(f"{save_path} already done")
        else:
            _, _, _, X_test, y_test = get_embeddings(sentencebertEncoder, train_reuters_, test_reuters_, inlier_topic, dataset_name, type_tac, anomaly_rate)
            print(f"data saved at : {save_path}")
            torch.save({"X_test": X_test, "y_test": y_test}, save_path)

In [ ]:
save_dir = "/home/2017025/ayouce01/Textual-Anomaly-Detection-Framework/Anomaly Detection Framework/Data"

dataset_name = 'agnews'
type_tac = 'fate' 
anomaly_rate = 0.1

inlier_topics = ['World', 'Sports', 'Business', 'Sci/Tech']

for inlier_topic in inlier_topics:
    
    if inlier_topic == 'Sci/Tech':
        inlier_topic = 'Sci-Tech'
    
    save_path_temp = os.path.join(save_dir, f"{dataset_name}/{inlier_topic}")
    os.makedirs(save_path_temp, exist_ok=True)
                                 
    save_path = os.path.join(save_path_temp, f"ds_train_{inlier_topic}.pt")
    
    if os.path.exists(save_path):
        print(f"{save_path} already done")
    else:
        if inlier_topic == 'Sci-Tech' :
            inlier_topic = 'Sci/Tech'
        _, _, X_inlier, _, _ = get_embeddings(sentencebertEncoder, train_agnews_, test_agnews_, inlier_topic, dataset_name, type_tac, anomaly_rate, 'train')
        print(f"data saved at : {save_path}")
        torch.save({"X_inlier": X_inlier}, save_path)

In [ ]:
save_dir = "/home/2017025/ayouce01/Textual-Anomaly-Detection-Framework/Anomaly Detection Framework/Data"

dataset_name = 'agnews'
type_tac = 'fate' 
anomaly_rate = 0.1

inlier_topics = ['World', 'Sports', 'Business', 'Sci/Tech']
nb_runs = 10

for run in range(nb_runs):

    for inlier_topic in inlier_topics:
        
        if inlier_topic == 'Sci/Tech':
            inlier_topic = 'Sci-Tech'

        save_path_temp = os.path.join(save_dir, f"{dataset_name}/{inlier_topic}/run{run+1}")
        os.makedirs(save_path_temp, exist_ok=True)

        save_path = os.path.join(save_path_temp, f"ds_test_{inlier_topic}_run{run+1}.pt")
        
        if os.path.exists(save_path):
            print(f"{save_path} already done")
        else:
            if inlier_topic == 'Sci-Tech' :
                inlier_topic = 'Sci/Tech'
            _, _, _, X_test, y_test = get_embeddings(sentencebertEncoder, train_agnews_, test_agnews_, inlier_topic, dataset_name, type_tac, anomaly_rate, 'test')
            print(f"data saved at : {save_path}")
            torch.save({"X_test": X_test, "y_test": y_test}, save_path)

In [ ]:
save_dir = "/home/2017025/ayouce01/Textual-Anomaly-Detection-Framework/Anomaly Detection Framework/Data"

dataset_name = 'dbpedia14'
type_tac = 'pantin' 
anomaly_rate = 0.1

inlier_topics = ["Company", "Educational Institution", "Artist", "Athlete", "Office Holder", "Mean Of Transportation", "Building", "Natural Place", "Village", "Animal", "Plant", "Album", "Film", "Written Work"]


for inlier_topic in inlier_topics:
    
    save_path_temp = os.path.join(save_dir, f"{dataset_name}/{inlier_topic}")
    os.makedirs(save_path_temp, exist_ok=True)
                                 
    save_path = os.path.join(save_path_temp, f"ds_train_{inlier_topic}.pt")
    
    if os.path.exists(save_path):
        print(f"{save_path} already done")
    else:
        _, _, X_inlier, _, _ = get_embeddings(sentencebertEncoder, train_dbpedia14_, test_dbpedia14_, inlier_topic, dataset_name, type_tac, anomaly_rate, 'train', 'content')
        print(f"data saved at : {save_path}")
        torch.save({"X_inlier": X_inlier}, save_path)

In [ ]:
save_dir = "/home/2017025/ayouce01/Textual-Anomaly-Detection-Framework/Anomaly Detection Framework/Data"

dataset_name = 'dbpedia14'
type_tac = 'pantin' 
anomaly_rate = 0.1

inlier_topics = ["Company", "Educational Institution", "Artist", "Athlete", "Office Holder", "Mean Of Transportation", "Building", "Natural Place", "Village", "Animal", "Plant", "Album", "Film", "Written Work"]
nb_runs = 10

for run in range(nb_runs):

    for inlier_topic in inlier_topics:

        save_path_temp = os.path.join(save_dir, f"{dataset_name}/{inlier_topic}/run{run+1}")
        os.makedirs(save_path_temp, exist_ok=True)

        save_path = os.path.join(save_path_temp, f"ds_test_{inlier_topic}_run{run+1}.pt")
        
        if os.path.exists(save_path):
            print(f"{save_path} already done")
        else:
            _, _, _, X_test, y_test = get_embeddings(sentencebertEncoder, train_dbpedia14_, test_dbpedia14_, inlier_topic, dataset_name, type_tac, anomaly_rate, 'test', 'content')
            print(f"data saved at : {save_path}")
            torch.save({"X_test": X_test, "y_test": y_test}, save_path)

### Loading Data

In [ ]:
def load_data_inlier(dataset_name, inlier_topic, save_dir = "/home/2017025/ayouce01/Textual-Anomaly-Detection-Framework/Anomaly Detection Framework/Data"):

    path = os.path.join(save_dir, f"{dataset_name}/{inlier_topic}/ds_train_{inlier_topic}.pt")

    return torch.load(path)['X_inlier']

def load_data_test(dataset_name, inlier_topic, n_run, save_dir = "/home/2017025/ayouce01/Textual-Anomaly-Detection-Framework/Anomaly Detection Framework/Data"):

    path = os.path.join(save_dir, f"{dataset_name}/{inlier_topic}/run{n_run}/ds_test_{inlier_topic}_run{n_run}.pt")
    ds = torch.load(path)
    
    return ds['X_test'], ds['y_test']

In [ ]:
inlier_topic = 'crude'
dataset_name = 'reuters'
save_dir = "/home/2017025/ayouce01/Textual-Anomaly-Detection-Framework/Anomaly Detection Framework/Data"
n_run = 7

X_inlier = load_data_inlier(dataset_name, inlier_topic, save_dir)
X_test, y_test = load_data_test(dataset_name, inlier_topic, n_run, save_dir)

In [ ]:
X_inlier.shape, X_test.shape, y_test.shape

(torch.Size([98, 768]), torch.Size([36, 768]), (36,))

In [ ]:
from sklearn.metrics.pairwise import pairwise_distances

ocsvm_kwargs = {
        "nu": 0.1,
        "kernel": 'rbf',
        # "gamma": 1 / (np.max(pairwise_distances(X_inlier.cpu())) ** 2)
        "gamma": 'scale'
        }
clf, _, _ = ocsvm.One_Class_SVM(X_inlier.cpu().detach(), ocsvm_kwargs)

_ = clf.predict(X_test.cpu().detach())           
scores_test = clf.decision_function(X_test.cpu().detach())

auc, ap, fpr95 = ev.evaluation(y_test, scores_test, verbose=False)
print(f"OCSVM --> AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")

OCSVM --> AUC: 0.9228 | FPR@95: 0.3670 | AP: 0.6335


In [ ]:
clf = fit_ae(X_inlier)
test_ae(X_test, y_test, clf)

cae = fit_rsrae(X_inlier)
auc, fpr95, ap = test_rsrae(X_test, y_test, cae)

In [ ]:
tccm = fit_tccm(X_inlier)
auc, fpr95, ap = test_tccm(X_test, y_test, tccm)

In [ ]:
for _ in range(1):

    batch_size = 32
    # batch_size = study.best_params['batch_size']
    X_inlier_dl = DataLoader(TensorDataset(X_inlier), batch_size=batch_size, shuffle=True)

    input_dim = X_inlier.shape[1]
    latent_dim = 256
    sinu = False
    batchnorm = False
    dropout = 0.1
    lr = 1e-3
    # lr = study.best_params['lr']
    weight_decay = 0
    # weight_decay = study.best_params['weight_decay']
    n_epochs = 30
    # n_epochs = study.best_params['n_epochs']


    target = X_inlier.cpu()
    source = 'sphere'
    # source = study.best_params['source']


    flow_model = flow_matching.FlowMatching(source, target, input_dim, latent_dim, sinu, dropout, batchnorm, device).to(device)
    optimizer = torch.optim.Adam(flow_model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    fm_trainer = flow_matching.FlowMatchingTrainer(flow_model, verbose=True)

    taac = time.time()

    flow_model_trained = fm_trainer.train(X_inlier_dl, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam')

    tiic = time.time()

    print(f"\nFM finishing... after {(tiic-taac)/60:.3f} mn")

    auc, fpr95, ap = fm_trainer.test(X_test, y_test, score_type='norm', solver_type='midpoint', n_steps=30)

 step 0 -> loss : 0.00231
 step 6 -> loss : 0.00199
 step 12 -> loss : 0.00214
 step 18 -> loss : 0.00197
 step 24 -> loss : 0.00172

FM finishing... after 0.006 mn
FM --> AUC: 0.7783 | FPR@95: 0.9541 | AP: 0.4357


In [ ]:
save_dir = "/home/2017025/ayouce01/Textual-Anomaly-Detection-Framework/Anomaly Detection Framework/Data"
file_path_hyp = "/home/2017025/ayouce01/Textual-Anomaly-Detection-Framework/Anomaly Detection Framework/Results/hyperparams.txt"

dataset_name = 'agnews'
# inlier_topics = ['computer','recreation', 'science', 'miscellaneous', 'politics', 'religion']
# inlier_topics = ['science', 'computer']
inlier_topics = ['World', 'Sports', 'Business', 'Sci-Tech']
# inlier_topics = ['earn', 'trade', 'acq', 'money-fx', 'crude', 'ship', 'interest']
# inlier_topics = ['acq', 'money-fx', 'crude', 'ship', 'interest']


for inlier_topic in inlier_topics:
    
    list_auc_fm = []
    list_fpr_fm = []
    list_ap_fm = []
    list_time_fm = []

    list_auc_ocsvm = []
    list_fpr_ocsvm = []
    list_ap_ocsvm = []
    list_time_ocsvm = []

    list_auc_rsrae = []
    list_fpr_rsrae = []
    list_ap_rsrae = []
    list_time_rsrae = []

    list_auc_ae = []
    list_fpr_ae = []
    list_ap_ae = []
    list_time_ae = []
    
    list_auc_tccm = []
    list_fpr_tccm = []
    list_ap_tccm = []
    list_time_tccm = []
        
    # print("---------------------------------")
    # print(f"-----------{inlier_topic}------------")
    # print("---------------------------------\n")
     
    X_inlier = load_data_inlier(dataset_name, inlier_topic, save_dir)
    hyp = load_hyperparams(dataset_name, inlier_topic, file_path_hyp)
    
    for n_run in range(1,11):
        
        # print("///////////////////////////////")
        # print(f"////////Run {n_run}////////")
        # print("///////////////////////////////\n")
           
        ################################################
        ################# LOADING DATA #################
        ################################################  
        
        X_test, y_test = load_data_test(dataset_name, inlier_topic, n_run, save_dir)
        
        #########################################
        ################# OCSVM #################
        #########################################  

        ocsvm_kwargs = {
            "nu": 0.1,
            "kernel": 'rbf',
            "gamma": 'scale'
            }
        
        taac = time.time()
        clf, _, _ = ocsvm.One_Class_SVM(X_inlier.cpu(), ocsvm_kwargs)
        tiic = time.time()

        print(f"OCSVM finishing... after {(tiic-taac)/60:.3f} mn")
        # list_time_ocsvm.append((tiic-taac)/60)

        _ = clf.predict(X_test.cpu().detach())           
        scores_test = clf.decision_function(X_test.cpu())

        auc_ocsvm, ap_ocsvm, fpr95_ocsvm = ev.evaluation(y_test, scores_test, verbose=False)
        # print(f"\nOCSVM --> AUC: {auc_ocsvm:.4f} | FPR@95: {fpr95_ocsvm:.4f} | AP: {ap_ocsvm:.4f}\n")

        list_auc_ocsvm.append(auc_ocsvm)
        list_fpr_ocsvm.append(fpr95_ocsvm)    
        list_ap_ocsvm.append(ap_ocsvm)
        list_time_ocsvm.append((tiic-taac))
        
        # #########################################
        # ################# RSRAE #################
        # #########################################  
        taac = time.time()
        cae = fit_rsrae(X_inlier)
        tiic = time.time()
        
        print(f"\nRSRAE finishing... after {(tiic-taac)/60:.3f} mn")
        list_time_rsrae.append((tiic-taac)/60)

        auc_rsrae, fpr95_rsrae, ap_rsrae = test_rsrae(X_test, y_test, cae)

        list_auc_rsrae.append(auc_rsrae)
        list_fpr_rsrae.append(fpr95_rsrae)    
        list_ap_rsrae.append(ap_rsrae) 
        list_time_rsrae.append((tiic-taac))

#         ######################################
#         ################# AE #################
#         ######################################
        taac = time.time()
        clf = fit_ae(X_inlier)
        tiic = time.time()
        
                
        # print(f"\nAE finishing... after {(tiic-taac)/60:.3f} mn")
        list_time_ae.append((tiic-taac)/60)
        
        auc_ae, fpr95_ae, ap_ae = test_ae(X_test, y_test, clf)

        list_auc_ae.append(auc_ae)
        list_fpr_ae.append(fpr95_ae)    
        list_ap_ae.append(ap_ae)  
        list_time_ae.append((tiic-taac))
        
#         #################################################
#         ################# Flow Matching #################
#         #################################################  
        
        # # batch_size = 2048
        # batch_size = hyp['batch_size']
        # # batch_size = study.best_params['batch_size']
        # X_inlier_dl = DataLoader(TensorDataset(X_inlier), batch_size=batch_size, shuffle=True)

        # input_dim = X_inlier.shape[1]
        # # latent_dim = 256
        # latent_dim = hyp['latent_dim']
        # sinu = False
        # batchnorm = False
        # # dropout = 0.1
        # dropout = hyp['dropout']
        # # lr = 1e-1
        # lr = hyp['lr']
        # # lr = study.best_params['lr']
        # # weight_decay = 1e-3
        # weight_decay = hyp['weight_decay']
        # # weight_decay = study.best_params['weight_decay']
        # # n_epochs = 30
        # n_epochs = hyp['n_epochs']
        # # n_epochs = study.best_params['n_epochs']


        # target = X_inlier.cpu()
        # # source = 'sphere'
        # source = hyp['source']
        # # source = study.best_params['source']


        # flow_model = flow_matching.FlowMatching(source, target, input_dim, latent_dim, sinu, dropout, batchnorm, device).to(device)
        # optimizer = torch.optim.Adam(flow_model.parameters(), lr=lr, weight_decay=weight_decay)
        # loss_fn = nn.MSELoss()

        # fm_trainer = flow_matching.FlowMatchingTrainer(flow_model, verbose=False)

        # taac = time.time()

        # flow_model_trained = fm_trainer.train(X_inlier_dl, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam')

        # tiic = time.time()

        # # print(f"\nFM finishing... after {(tiic-taac)/60:.3f} mn")

        # auc, fpr95, ap = fm_trainer.test(X_test, y_test, score_type='norm', solver_type='midpoint', n_steps=10)
        
        # list_auc_fm.append(auc)
        # list_fpr_fm.append(fpr95)    
        # list_ap_fm.append(ap)
        # list_time_fm.append((tiic-taac))

        ########################################
        ################# TCCM #################
        ########################################  
        taac = time.time()
        tccm = fit_tccm(X_inlier)
        tiic = time.time()
        
        auc_tccm, fpr95_tccm, ap_tccm = test_tccm(X_test, y_test, tccm)
        
        list_auc_tccm.append(auc_tccm)
        list_fpr_tccm.append(fpr95_tccm)    
        list_ap_tccm.append(ap_tccm)  
        list_time_tccm.append((tiic-taac))  
        
    # print(inlier_topic ,np.mean(list_auc_fm))
    # save_results(
    #     dataset_name=dataset_name, inlier_topic=inlier_topic ,type_emb="sentence_bert" ,ad_model="flow-matching",
    #     auc_mean=np.mean(list_auc_fm), ap_mean=np.mean(list_ap_fm),fpr_mean=np.mean(list_fpr_fm),
    #     auc_std = np.std(list_auc_fm),ap_std =  np.std(list_ap_fm),fpr_std = np.std(list_fpr_fm),
    #     train_time = np.mean(list_time_fm),overwrite='naive'
    #     )

        
    save_results(
        dataset_name=dataset_name, inlier_topic=inlier_topic ,type_emb="sentence_bert" ,ad_model="AE",
        auc_mean=np.mean(list_auc_ae), ap_mean=np.mean(list_ap_ae),fpr_mean=np.mean(list_fpr_ae),
        auc_std = np.std(list_auc_ae),ap_std =  np.std(list_ap_ae),fpr_std = np.std(list_fpr_ae),
        train_time = np.array(list_time_ae)[np.array(list_time_ae) > 1].mean(),overwrite='naive'
        )

    save_results(
        dataset_name=dataset_name, inlier_topic=inlier_topic ,type_emb="sentence_bert" ,ad_model="RSRAE",
        auc_mean=np.mean(list_auc_rsrae), ap_mean=np.mean(list_ap_rsrae),fpr_mean=np.mean(list_fpr_rsrae),
        auc_std = np.std(list_auc_rsrae),ap_std =  np.std(list_ap_rsrae),fpr_std = np.std(list_fpr_rsrae),
        train_time = np.array(list_time_rsrae)[np.array(list_time_rsrae) > 2].mean(), overwrite='naive'
    )

    save_results(
        dataset_name=dataset_name, inlier_topic=inlier_topic ,type_emb="sentence_bert" ,ad_model="ocsvm",
        auc_mean=np.mean(list_auc_ocsvm), ap_mean=np.mean(list_ap_ocsvm),fpr_mean=np.mean(list_fpr_ocsvm),
        auc_std = np.std(list_auc_ocsvm),ap_std =  np.std(list_ap_ocsvm),fpr_std = np.std(list_fpr_ocsvm),
        train_time = np.array(list_time_ocsvm)[np.array(list_time_ocsvm) > 4].mean(), overwrite='naive'
    )

    save_results(
        dataset_name=dataset_name, inlier_topic=inlier_topic ,type_emb="sentence_bert" ,ad_model="TCCM",
        auc_mean=np.mean(list_auc_tccm), ap_mean=np.mean(list_ap_tccm),fpr_mean=np.mean(list_fpr_tccm),
        auc_std = np.std(list_auc_tccm),ap_std =  np.std(list_ap_tccm),fpr_std = np.std(list_fpr_tccm),
        train_time = np.mean(list_time_tccm), overwrite='naive'
    )


OCSVM finishing... after 2.113 mn


I0000 00:00:1765200878.987448 2302825 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 78725 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:48:00.0, compute capability: 8.0


 epoch 50/100 : loss = 0.955
 epoch 100/100 : loss = 0.955

RSRAE finishing... after 0.667 mn
RSRAE --> AUC: 0.9182 | FPR@95: 0.2363 | AP: 0.4679
AE --> AUC: 0.9352 | FPR@95: 0.1982 | AP: 0.5817
TCCM --> AUC: 0.9421 | FPR@95: 0.2240 | AP: 0.6344
OCSVM finishing... after 2.821 mn


I0000 00:00:1765201188.262719 2302825 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 78725 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:48:00.0, compute capability: 8.0


 epoch 50/100 : loss = 0.955
 epoch 100/100 : loss = 0.955

RSRAE finishing... after 0.690 mn
RSRAE --> AUC: 0.9009 | FPR@95: 0.2713 | AP: 0.4395
AE --> AUC: 0.9186 | FPR@95: 0.3006 | AP: 0.5171
TCCM --> AUC: 0.9273 | FPR@95: 0.2801 | AP: 0.5698
OCSVM finishing... after 2.118 mn


...

I0000 00:00:1765206389.566516 2302825 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 78725 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:48:00.0, compute capability: 8.0


 epoch 50/100 : loss = 0.954
 epoch 100/100 : loss = 0.954

RSRAE finishing... after 0.672 mn
RSRAE --> AUC: 0.9051 | FPR@95: 0.4380 | AP: 0.5747
AE --> AUC: 0.9363 | FPR@95: 0.2602 | AP: 0.6555
TCCM --> AUC: 0.9340 | FPR@95: 0.2737 | AP: 0.6737
OCSVM finishing... after 2.234 mn


I0000 00:00:1765206670.019801 2302825 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 78725 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:48:00.0, compute capability: 8.0


 epoch 50/100 : loss = 0.954
 epoch 100/100 : loss = 0.954

RSRAE finishing... after 0.667 mn
RSRAE --> AUC: 0.8837 | FPR@95: 0.4865 | AP: 0.5349
AE --> AUC: 0.9135 | FPR@95: 0.4439 | AP: 0.6495


KeyboardInterrupt: 

In [ ]:
# save_results(
#     dataset_name=dataset_name, inlier_topic=inlier_topic ,type_emb="sentence_bert" ,ad_model="flow-matching",
#     auc_mean=np.mean(list_auc_fm), ap_mean=np.mean(list_ap_fm),fpr_mean=np.mean(list_fpr_fm),
#     auc_std = np.std(list_auc_fm),ap_std =  np.std(list_ap_fm),fpr_std = np.std(list_fpr_fm),
#     train_time = np.mean(list_time_fm),overwrite='naive'
# )

save_results(
    dataset_name=dataset_name, inlier_topic=inlier_topic ,type_emb="sentence_bert" ,ad_model="AE",
    auc_mean=np.mean(list_auc_ae), ap_mean=np.mean(list_ap_ae),fpr_mean=np.mean(list_fpr_ae),
    auc_std = np.std(list_auc_ae),ap_std =  np.std(list_ap_ae),fpr_std = np.std(list_fpr_ae),
    train_time = np.array(list_time_ae)[np.array(list_time_ae) > 1].mean(),overwrite='naive'
)

save_results(
    dataset_name=dataset_name, inlier_topic=inlier_topic ,type_emb="sentence_bert" ,ad_model="RSRAE",
    auc_mean=np.mean(list_auc_rsrae), ap_mean=np.mean(list_ap_rsrae),fpr_mean=np.mean(list_fpr_rsrae),
    auc_std = np.std(list_auc_rsrae),ap_std =  np.std(list_ap_rsrae),fpr_std = np.std(list_fpr_rsrae),
    train_time = np.array(list_time_rsrae)[np.array(list_time_rsrae) > 2].mean(), overwrite='naive'
)

save_results(
    dataset_name=dataset_name, inlier_topic=inlier_topic ,type_emb="sentence_bert" ,ad_model="ocsvm",
    auc_mean=np.mean(list_auc_ocsvm), ap_mean=np.mean(list_ap_ocsvm),fpr_mean=np.mean(list_fpr_ocsvm),
    auc_std = np.std(list_auc_ocsvm),ap_std =  np.std(list_ap_ocsvm),fpr_std = np.std(list_fpr_ocsvm),
    train_time = np.array(list_time_ocsvm)[np.array(list_time_ocsvm) > 4].mean(), overwrite='naive'
)

save_results(
    dataset_name=dataset_name, inlier_topic=inlier_topic ,type_emb="sentence_bert" ,ad_model="TCCM",
    auc_mean=np.mean(list_auc_tccm), ap_mean=np.mean(list_ap_tccm),fpr_mean=np.mean(list_fpr_tccm),
    auc_std = np.std(list_auc_tccm),ap_std =  np.std(list_ap_tccm),fpr_std = np.std(list_fpr_tccm),
    train_time = np.mean(list_time_tccm), overwrite='naive'
)

create_tables()

# save_hyperparameters(dataset_name, inlier_topic,
#                          batch_size=2048,
#                          latent_dim=256,
#                          sinu=False,
#                          batchnorm=False,
#                          dropout=0.1,
#                          lr=1e-1,
#                          weight_decay=1e-3,
#                          n_epochs=30,
#                          target=X_inlier.shape,
#                          source='sphere',
#                          save_dir="Results")

In [ ]:
np.mean(list_auc_fm), np.mean(list_auc_ae), np.mean(list_auc_ocsvm), np.mean(list_auc_rsrae), np.mean(list_auc_tccm)

In [495]:
create_tables()

Lecture des fichiers dans chaque dataset...
Fichier LaTeX généré : /home/2017025/ayouce01/Textual-Anomaly-Detection-Framework/Anomaly Detection Framework/Results/tables.tex


In [ ]:
save_dir = "/home/2017025/ayouce01/Textual-Anomaly-Detection-Framework/Anomaly Detection Framework/Data"
dataset_name = '20newsgroups'
inlier_topics = ['computer', 'recreation', 'science', 'miscellaneous', 'politics', 'religion']

for inlier_topic in inlier_topics:
    
    list_auc_fm = []
    list_fpr_fm = []
    list_ap_fm = []
    list_time_fm = []

    list_auc_ocsvm = []
    list_fpr_ocsvm = []
    list_ap_ocsvm = []
    list_time_ocsvm = []

    list_auc_rsrae = []
    list_fpr_rsrae = []
    list_ap_rsrae = []
    list_time_rsrae = []

    list_auc_ae = []
    list_fpr_ae = []
    list_ap_ae = []
    list_time_ae = []
    
    print("---------------------------------")
    print(f"-----------{inlier_topic}------------")
    print("---------------------------------\n")
    
    
    X_inlier = load_X_inlier(dataset_name, inlier_topic, save_dir)
    
    for n_run in range(1,11):
        
        print("///////////////////////////////")
        print(f"////////Run {n_run}////////")
        print("///////////////////////////////\n")
           
        ################################################
        ################# LOADING DATA #################
        ################################################  
        
        X_test, y_test = load_X_test(dataset_name, inlier_topic, n_run, save_dir)
        
        #########################################
        ################# OCSVM #################
        #########################################  

        ocsvm_kwargs = {
            "nu": 0.1,
            "kernel": 'rbf',
            "gamma": 'scale'
            }
        
        taac = time.time()
        clf, _, _ = ocsvm.One_Class_SVM(X_inlier.cpu(), ocsvm_kwargs)
        tiic = time.time()

        print(f"OCSVM finishing... after {(tiic-taac)/60:.3f} mn")
        list_time_ocsvm.append((tiic-taac)/60)

        _ = clf.predict(X_test.cpu().detach())           
        scores_test = clf.decision_function(X_test.cpu())

        auc_ocsvm, ap_ocsvm, fpr95_ocsvm = ev.evaluation(y_test, scores_test, verbose=False)
        print(f"\nOCSVM --> AUC: {auc_ocsvm:.4f} | FPR@95: {fpr95_ocsvm:.4f} | AP: {ap_ocsvm:.4f}\n")

        list_auc_ocsvm.append(auc_ocsvm)
        list_fpr_ocsvm.append(fpr95_ocsvm)    
        list_ap_ocsvm.append(ap_ocsvm)  
        
        #########################################
        ################# RSRAE #################
        #########################################  
        taac = time.time()
        cae = fit_rsrae(X_inlier)
        tiic = time.time()
        
        print(f"\nRSRAE finishing... after {(tiic-taac)/60:.3f} mn")
        list_time_rsrae.append((tiic-taac)/60)

        auc_rsrae, fpr95_rsrae, ap_rsrae = test_rsrae(X_test, y_test, cae)

        list_auc_rsrae.append(auc_rsrae)
        list_fpr_rsrae.append(fpr95_rsrae)    
        list_ap_rsrae.append(ap_rsrae)  


        ######################################
        ################# AE #################
        ######################################
        taac = time.time()
        clf = fit_ae(X_inlier)
        tiic = time.time()
        
                
        print(f"\nAE finishing... after {(tiic-taac)/60:.3f} mn")
        list_time_ae.append((tiic-taac)/60)
        
        auc_ae, fpr95_ae, ap_ae = test_ae(X_test, y_test, clf)

        list_auc_ae.append(auc_ae)
        list_fpr_ae.append(fpr95_ae)    
        list_ap_ae.append(ap_ae)  
        
    save_results(
    dataset_name=dataset_name,
    inlier_topic=inlier_topic,
    type_emb="sentence_bert",
    ad_model="RSRAE",
    auc_mean=np.mean(list_auc_rsrae),
    ap_mean=np.mean(list_ap_rsrae),
    fpr_mean=np.mean(list_fpr_rsrae),
    auc_std = np.std(list_auc_rsrae),
    ap_std =  np.std(list_ap_rsrae),
    fpr_std = np.std(list_fpr_rsrae),
    overwrite='naive'
)

    save_results(
        dataset_name=dataset_name,
        inlier_topic=inlier_topic,
        type_emb="sentence_bert",
        ad_model="AE",
        auc_mean=np.mean(list_auc_ae),
        ap_mean=np.mean(list_ap_ae),
        fpr_mean=np.mean(list_fpr_ae),
        auc_std = np.std(list_auc_ae),
        ap_std =  np.std(list_ap_ae),
        fpr_std = np.std(list_fpr_ae),
        overwrite='naive'
    )

    save_results(
        dataset_name=dataset_name,
        inlier_topic=inlier_topic,
        type_emb="sentence_bert",
        ad_model="ocsvm",
        auc_mean=np.mean(list_auc_ocsvm),
        ap_mean=np.mean(list_ap_ocsvm),
        fpr_mean=np.mean(list_fpr_ocsvm),
        auc_std = np.std(list_auc_ocsvm),
        ap_std =  np.std(list_ap_ocsvm),
        fpr_std = np.std(list_fpr_ocsvm),
        overwrite='naive'
    )


#         ################################################
#         ################ FLOW MATCHING #################
#         ################################################

#         batch_size = 512
#         # batch_size = study.best_params['batch_size']
#         X_inlier_dl = DataLoader(TensorDataset(X_inlier), batch_size=batch_size, shuffle=True)

#         input_dim = X_inlier.shape[1]
#         latent_dim = 256
#         sinu = False
#         lr = 1e-4
#         # lr = study.best_params['lr']
#         weight_decay = 1e-4
#         # weight_decay = study.best_params['weight_decay']
#         n_epochs = 100
#         # n_epochs = study.best_params['n_epochs']


#         target = X_inlier.cpu()
#         source = 'sphere'
#         # source = study.best_params['source']


#         flow_model = flow_matching.FlowMatching(source, target, input_dim, latent_dim, sinu, device).to(device)
#         optimizer = torch.optim.Adam(flow_model.parameters(), lr=lr, weight_decay=weight_decay)
#         loss_fn = nn.MSELoss()

#         fm_trainer = flow_matching.FlowMatchingTrainer(flow_model, verbose=True)

#         taac = time.time()

#         flow_model_trained = fm_trainer.train(X_inlier_dl, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam')

#         tiic = time.time()

#         print(f"\nFM finishing... after {(tiic-taac)/60:.3f} mn")

#         auc, fpr95, ap = fm_trainer.test(X_test, y_test, score_type='norm', solver_type='midpoint', n_steps=10)

#         list_auc_fm.append(auc)
#         list_fpr_fm.append(fpr95)    
#         list_ap_fm.append(ap)
          


    

In [ ]:
create_tables()

### Investigation

In [ ]:
inlier_topic = 'acq'
dataset_name = 'reuters'
type_tac = 'ruff' 
anomaly_rate = 0.1

train_inlier_reuters_acq, test_reuters_acq, X_inlier_acq, X_test_acq, y_test_acq = get_embeddings(train_reuters_, test_reuters_, inlier_topic, dataset_name, type_tac, anomaly_rate)

In [ ]:
inlier_topic = 'crude'
dataset_name = 'reuters'
type_tac = 'ruff' 
anomaly_rate = 0.1

train_inlier_reuters_crude, test_reuters_crude, X_inlier_crude, X_test_crude, y_test_crude = get_embeddings(train_reuters_, test_reuters_, inlier_topic, dataset_name, type_tac, anomaly_rate)

In [ ]:
inlier_topic = 'earn'
dataset_name = 'reuters'
type_tac = 'ruff' 
anomaly_rate = 0.1

train_inlier_reuters_earn, test_reuters_earn, X_inlier_earn, X_test_earn, y_test_earn = get_embeddings(train_reuters_, test_reuters_, inlier_topic, dataset_name, type_tac, anomaly_rate)

In [ ]:
inlier_topic = 'ship'
dataset_name = 'reuters'
type_tac = 'ruff' 
anomaly_rate = 0.1

train_inlier_reuters_ship, test_reuters_ship, X_inlier_ship, X_test_ship, y_test_ship = get_embeddings(train_reuters_, test_reuters_, inlier_topic, dataset_name, type_tac, anomaly_rate)

In [ ]:
print(X_inlier_acq.mean())
print(X_inlier_crude.mean())
print(X_inlier_earn.mean())
print(X_inlier_ship.mean())

print()

print(X_inlier_acq.max())
print(X_inlier_crude.max())
print(X_inlier_earn.max())
print(X_inlier_ship.max())

print()

print(X_inlier_acq.min())
print(X_inlier_crude.min())
print(X_inlier_earn.min())
print(X_inlier_ship.min())

In [ ]:
X_embedded = TSNE(n_components=3, learning_rate='auto',
                  init='random', perplexity=3).fit_transform(X_inlier_acq.cpu())

fig = plt.figure()
ax = fig.add_subplot(projection='3d')

ax.scatter(X_embedded[:,0], X_embedded[:,1], X_embedded[:,2], color='crimson')

In [ ]:
X_embedded = TSNE(n_components=3, learning_rate='auto',
                  init='random', perplexity=3).fit_transform(X_inlier_crude.cpu())

fig = plt.figure()
ax = fig.add_subplot(projection='3d')

ax.scatter(X_embedded[:,0], X_embedded[:,1], X_embedded[:,2], color='crimson')

In [ ]:
X_embedded = TSNE(n_components=3, learning_rate='auto',
                  init='random', perplexity=3).fit_transform(X_inlier_earn.cpu())

fig = plt.figure()
ax = fig.add_subplot(projection='3d')

ax.scatter(X_embedded[:,0], X_embedded[:,1], X_embedded[:,2], color='crimson')

In [ ]:
X_embedded = TSNE(n_components=3, learning_rate='auto',
                  init='random', perplexity=3).fit_transform(X_inlier_ship.cpu())

fig = plt.figure()
ax = fig.add_subplot(projection='3d')

ax.scatter(X_embedded[:,0], X_embedded[:,1], X_embedded[:,2], color='crimson')

In [ ]:
z = torch.randn(1500, 384)
z = z / z.norm(dim=1, keepdim=True)
noise = torch.randn_like(z) * 0.25
X__ = Tensor(z + noise)

X_embedded = TSNE(n_components=3, learning_rate='auto',
                  init='random', perplexity=3).fit_transform(X__)

fig = plt.figure()
ax = fig.add_subplot(projection='3d')

ax.scatter(X_embedded[:,0], X_embedded[:,1], X_embedded[:,2], color='crimson')

In [ ]:
target_centroid = X_inlier_acq.mean(dim=0).cpu()
target_cov_mat = torch.cov(X_inlier_acq.T).cpu()

eps = 1e-2
cov_reg = target_cov_mat + eps * torch.eye(target_cov_mat.size(0))

distri = torch.distributions.MultivariateNormal(target_centroid, covariance_matrix=cov_reg)
z = distri.sample((1500,))

z = z / z.norm(dim=1, keepdim=True)
noise = torch.randn_like(z) * 0.25
X__ = Tensor(z + noise)

X_embedded = TSNE(n_components=3, learning_rate='auto',
                  init='random', perplexity=3).fit_transform(X__)

fig = plt.figure()
ax = fig.add_subplot(projection='3d')

ax.scatter(X_embedded[:,0], X_embedded[:,1], X_embedded[:,2], color='crimson')

In [ ]:
target_centroid = X_inlier_crude.mean(dim=0).cpu()
target_cov_mat = torch.cov(X_inlier_crude.T).cpu()

eps = 1e-2
cov_reg = target_cov_mat + eps * torch.eye(target_cov_mat.size(0))

distri = torch.distributions.MultivariateNormal(target_centroid, covariance_matrix=cov_reg)
z = distri.sample((1500,))

z = z / z.norm(dim=1, keepdim=True)
noise = torch.randn_like(z) * 0.25
X__ = Tensor(z + noise)

print(X__.mean())
print(X__.max())
print(X__.min())

print("-----------------")

z = torch.randn(1500, 384)
z = z / z.norm(dim=1, keepdim=True)
noise = torch.randn_like(z) * 0.25
X__ = Tensor(z + noise)

print(X__.mean())
print(X__.max())
print(X__.min())

## FM 

### Hyperparameters Search

In [ ]:
batch_size_default = 32
X_inlier_dl = DataLoader(TensorDataset(X_inlier), batch_size=batch_size_default, shuffle=True)
input_dim = X_inlier.shape[1]
latent_dim = 256
sinu = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def objective(trial):

    # batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
    # n_epochs = trial.suggest_int("n_epochs", 1000, 2000, step=100)
    n_epochs = trial.suggest_int("n_epochs", 50, 500, step=50)
    source = trial.suggest_categorical("source", ["sphere", "sphere-noised", "uniform"])
    # source = trial.suggest_categorical("source", ["gaussian", "sphere", "sphere-noised"])
    lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
    weight_decay = trial.suggest_float("weight_decay", 0, 1e-3, log=False)

    dl_train = DataLoader(TensorDataset(X_inlier), batch_size=batch_size, shuffle=True)

    flow_model = flow_matching.FlowMatching(source, X_inlier.cpu(), input_dim, latent_dim, sinu, device).to(device)

    optimizer = torch.optim.Adam(flow_model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    print("\n############################################################")
    print(f"batch_size: {batch_size} | n_epochs:{n_epochs} | source: {source} | lr: {lr} | weight_decay: {weight_decay}\n")
    fm_trainer = flow_matching.FlowMatchingTrainer(flow_model, verbose=True)
    flow_model_trained = fm_trainer.train(dl_train, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam')
    

    auc, fpr95, ap = fm_trainer.test(X_val, y_val, score_type='norm', solver_type='midpoint', n_steps=10)

    score = auc + ap - fpr95
    
    print(f"FM --> AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f} | SCORE: {score: .4f}")  
    
#     ocsvm_kwargs = {
#         "nu": 0.1,
#         "kernel": 'rbf',
#         "gamma": 'scale'
#         }
#     clf, _, _ = ocsvm.One_Class_SVM(X_inlier.cpu().detach(), ocsvm_kwargs)

#     _ = clf.predict(X_test.cpu().detach())           
#     scores_val = clf.decision_function(X_val.cpu().detach())

#     auc, ap, fpr95 = ev.evaluation(y_val, scores_val, verbose=False)
#     print(f"OCSVM --> AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")
    
    return score


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15)  

print("Best hyperparameters:", study.best_params)
print("Best score:", study.best_value)

### Running

In [ ]:
# inlier_topics = ['computer', 'recreation', 'science', 'miscellaneous', 'politics', 'religion']
inlier_topics = ['World', 'Sports', 'Business', 'Sci/Tech']


for i, inlier_topic in enumerate(inlier_topics):
    
    print("---------------------------------")
    print(f"--------- {inlier_topic} ------------")
    print("---------------------------------")
          

    # inlier_topic = 'crude'
    dataset_name = 'agnews'
    type_tac = 'fate' 
    anomaly_rate = 0.1

    train_inlier_agnews, _, _, _ = train_test_val_split(train_agnews_, test_agnews_, inlier_topic, dataset_name, type_tac, anomaly_rate, False)

    model_name = 'all-MiniLM-L6-v2'
    sentencebertEncoder = embedding_encoder.EmbeddingEncoder(model_name, 'sentencebert', device)

    train_inlier_agnews_emb = sentencebertEncoder.forward(train_inlier_agnews)
    X_inlier = Tensor(train_inlier_agnews_emb['sbert_embeddings']).to(device)

    list_auc_fm = []
    list_fpr_fm = []
    list_ap_fm = []
    list_auc_ocsvm = []
    list_fpr_ocsvm = []
    list_ap_ocsvm = []
    list_auc_rsrae = []
    list_fpr_rsrae = []
    list_ap_rsrae = []
    list_auc_ae = []
    list_fpr_ae = []
    list_ap_ae = []

    for i in range(3):

        print("\n##################################")
        print(f"Loading Dataset for the run {i+1}")

        # inlier_topic = 'crude'
        dataset_name = 'agnews'
        type_tac = 'fate' 
        anomaly_rate = 0.1

        _, _, _, test_agnews = train_test_val_split(train_agnews_, test_agnews, inlier_topic, dataset_name, type_tac, anomaly_rate, False)

        # print("A sample for valset : ")
        # print(val_20ng[0]['text'])
        # print()
        # print("\nVALSET")
        # print(val_20ng.num_rows)
        # print()
        print("A sample for testset : ")
        print(test_agnews[-1]['text'][:50])
        print()
        # print("TESTSET")
        # print(test_20ng.num_rows)



        test_agnews_emb = sentencebertEncoder.forward(test_agnews)
        # val_reuters_emb = sentencebertEncoder.forward(val_reuters)

        # print(X_inlier.shape)

        X_test =  Tensor(test_agnews_emb['sbert_embeddings']).to(device)
        y_test = np.array(test_agnews_emb['anomaly_class'])
        # print(X_test.shape, y_test.shape)

        # X_val =  Tensor(val_reuters_emb['sbert_embeddings']).to(device)
        # y_val = np.array(val_reuters_emb['anomaly_class'])
        # print(X_val.shape, y_val.shape)

        #########################################
        ################# OCSVM #################
        #########################################  

    #     taac = time.time()

    #     ocsvm_kwargs = {
    #         "nu": 0.1,
    #         "kernel": 'rbf',
    #         "gamma": 'scale'
    #         }
    #     clf, _, _ = ocsvm.One_Class_SVM(X_inlier.cpu(), ocsvm_kwargs)

    #     tiic = time.time()

    #     print(f"OCSVM finishing... after {(tiic-taac)/60:.3f} mn")

    #     _ = clf.predict(X_test.cpu().detach())           
    #     scores_test = clf.decision_function(X_test.cpu())

    #     auc_ocsvm, ap_ocsvm, fpr95_ocsvm = ev.evaluation(y_test, scores_test, verbose=False)
    #     print(f"OCSVM --> AUC: {auc_ocsvm:.4f} | FPR@95: {fpr95_ocsvm:.4f} | AP: {ap_ocsvm:.4f}\n")

    #     list_auc_ocsvm.append(auc_ocsvm)
    #     list_fpr_ocsvm.append(fpr95_ocsvm)    
    #     list_ap_ocsvm.append(ap_ocsvm)  



        #########################################
        ################# RSRAE #################
        #########################################  
        cae = fit_rsrae(X_inlier)
        auc_rsrae, fpr95_rsrae, ap_rsrae = test_rsrae(X_test, y_test, cae)

        list_auc_rsrae.append(auc_rsrae)
        list_fpr_rsrae.append(fpr95_rsrae)    
        list_ap_rsrae.append(ap_rsrae)  


        ######################################
        ################# AE #################
        ######################################

        clf = fit_ae(X_inlier)
        auc_ae, fpr95_ae, ap_ae = test_ae(X_test, y_test, clf)

        list_auc_ae.append(auc_ae)
        list_fpr_ae.append(fpr95_ae)    
        list_ap_ae.append(ap_ae)  


        ################################################
        ################ FLOW MATCHING #################
        ################################################

    #     batch_size = 512
    #     # batch_size = study.best_params['batch_size']
    #     X_inlier_dl = DataLoader(TensorDataset(X_inlier), batch_size=batch_size, shuffle=True)

    #     input_dim = X_inlier.shape[1]
    #     latent_dim = 256
    #     sinu = False
    #     lr = 1e-4
    #     # lr = study.best_params['lr']
    #     weight_decay = 1e-4
    #     # weight_decay = study.best_params['weight_decay']
    #     n_epochs = 100
    #     # n_epochs = study.best_params['n_epochs']


    #     target = X_inlier.cpu()
    #     source = 'sphere'
    #     # source = study.best_params['source']


    #     flow_model = flow_matching.FlowMatching(source, target, input_dim, latent_dim, sinu, device).to(device)
    #     optimizer = torch.optim.Adam(flow_model.parameters(), lr=lr, weight_decay=weight_decay)
    #     loss_fn = nn.MSELoss()

    #     fm_trainer = flow_matching.FlowMatchingTrainer(flow_model, verbose=True)

    #     taac = time.time()

    #     flow_model_trained = fm_trainer.train(X_inlier_dl, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam')

    #     tiic = time.time()

    #     print(f"\nFM finishing... after {(tiic-taac)/60:.3f} mn")

    #     auc, fpr95, ap = fm_trainer.test(X_test, y_test, score_type='norm', solver_type='midpoint', n_steps=10)

    #     list_auc_fm.append(auc)
    #     list_fpr_fm.append(fpr95)    
    #     list_ap_fm.append(ap)
          
          
    save_results(
        dataset_name=dataset_name,
        inlier_topic=inlier_topic,
        type_emb="sentence_bert",
        ad_model="RSRAE",
        auc_mean=np.mean(list_auc_rsrae),
        ap_mean=np.mean(list_ap_rsrae),
        fpr_mean=np.mean(list_fpr_rsrae),
        auc_std = np.std(list_auc_rsrae),
        ap_std =  np.std(list_ap_rsrae),
        fpr_std = np.std(list_fpr_rsrae),
        overwrite='smart'
    )
                
    save_results(
        dataset_name=dataset_name,
        inlier_topic=inlier_topic,
        type_emb="sentence_bert",
        ad_model="AE",
        auc_mean=np.mean(list_auc_ae),
        ap_mean=np.mean(list_ap_ae),
        fpr_mean=np.mean(list_fpr_ae),
        auc_std = np.std(list_auc_ae),
        ap_std =  np.std(list_ap_ae),
        fpr_std = np.std(list_fpr_ae),
        overwrite='smart'
    )

In [ ]:
save_results(
    dataset_name=dataset_name,
    inlier_topic=inlier_topic,
    type_emb="sentence_bert",
    ad_model="flow-matching",
    auc_mean=np.mean(list_auc_fm),
    ap_mean=np.mean(list_ap_fm),
    fpr_mean=np.mean(list_fpr_fm),
    auc_std = np.std(list_auc_fm),
    ap_std =  np.std(list_ap_fm),
    fpr_std = np.std(list_fpr_fm),
    overwrite='smart'
)

In [ ]:
save_results(
    dataset_name=dataset_name,
    inlier_topic=inlier_topic,
    type_emb="sentence_bert",
    ad_model="ocsvm",
    auc_mean=np.mean(list_auc_ocsvm),
    ap_mean=np.mean(list_ap_ocsvm),
    fpr_mean=np.mean(list_fpr_ocsvm),
    auc_std = np.std(list_auc_ocsvm),
    ap_std =  np.std(list_ap_ocsvm),
    fpr_std = np.std(list_fpr_ocsvm),
    overwrite='smart'
)

In [ ]:
save_results(
    dataset_name=dataset_name,
    inlier_topic=inlier_topic,
    type_emb="sentence_bert",
    ad_model="RSRAE",
    auc_mean=np.mean(list_auc_rsrae),
    ap_mean=np.mean(list_ap_rsrae),
    fpr_mean=np.mean(list_fpr_rsrae),
    auc_std = np.std(list_auc_rsrae),
    ap_std =  np.std(list_ap_rsrae),
    fpr_std = np.std(list_fpr_rsrae),
    overwrite='smart'
)

In [ ]:
create_tables()

In [ ]:
inlier_topic = 'computer'
dataset_name = '20newsgroups'
type_tac = 'ruff' 
anomaly_rate = 0.1

train_inlier_reuters_acq, test_reuters_acq, X_inlier, X_test, y_test = get_embeddings(train_20ng_, test_20ng_, inlier_topic, dataset_name, type_tac, anomaly_rate)

In [ ]:
epochs_rsrae = 200
lr_rsrae = 1e-3

def objective(trial):

    epochs_rsrae = trial.suggest_int("n_epochs", 50, 500, step=50)
    lr_rsrae = trial.suggest_float("lr", 1e-3, 1e-1, log=True)
    
    print("\n############################################################")
    print(f"lr: {lr_rsrae} | n_epochs:{epochs_rsrae}\n")
    
    cae = fit_rsrae(X_inlier, lr_rsrae, epochs_rsrae)
    auc, fpr95, ap = test_rsrae(X_test, y_test, cae)

    score = auc + ap - fpr95
        
    return score

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)  

print("Best hyperparameters:", study.best_params)
print("Best score:", study.best_value)

In [ ]:
cae = fit_rsrae(X_inlier, 1e-3, 350)
test_rsrae(X_test, y_test, cae)

In [ ]:
clf = fit_ae(X_inlier)
test_ae(X_test, y_test, clf)

In [ ]:
for _ in range(1):

    batch_size = 64
    # batch_size = study.best_params['batch_size']
    X_inlier_dl = DataLoader(TensorDataset(X_inlier), batch_size=batch_size, shuffle=True)

    input_dim = X_inlier.shape[1]
    latent_dim = 256
    sinu = False
    lr = 1e-4
    # lr = study.best_params['lr']
    weight_decay = 1e-5
    # weight_decay = study.best_params['weight_decay']
    n_epochs = 300
    # n_epochs = study.best_params['n_epochs']

    target = X_inlier.cpu()
    source = 'sphere-noised'
    # source = study.best_params['source']

    flow_model = flow_matching.FlowMatching(source, target, input_dim, latent_dim, sinu, device).to(device)
    optimizer = torch.optim.Adam(flow_model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    fm_trainer = flow_matching.FlowMatchingTrainer(flow_model, verbose=True)

    flow_model_trained = fm_trainer.train(X_inlier_dl, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam')

    auc, fpr95, ap = fm_trainer.test(X_test, y_test, score_type='norm', solver_type='midpoint', n_steps=10)

## Baselines

### OCSVM

In [ ]:
ocsvm_kwargs = {
        "nu": 0.1,
        "kernel": 'rbf',
        "gamma": 'scale'
        }
clf, _, _ = ocsvm.One_Class_SVM(X_inlier.cpu().detach(), ocsvm_kwargs)

_ = clf.predict(X_test.cpu().detach())           
scores_test = clf.decision_function(X_test.cpu().detach())

auc, ap, fpr95 = ev.evaluation(y_test, scores_test, verbose=False)
print(f"AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")

### CVDD

In [ ]:
type_emb = 'glove'
# emb_model = 'distilbert-base-uncased'
attention_size = 150
n_attention_heads = 10
lr = 0.0001
lr_milestones = (40, 60)
n_epochs = 100
lambda_p = 1.0
alpha_scheduler = 'logarithmic'

In [ ]:
 dataset_name = '20newsgroups'
inlier_topic = 'computer'
type_tac = 'ruff'
anomaly_rate = 0.1

train_inlier_reuters, train_anomaly_reuters, val_reuters, test_reuters = train_test_val_split(train_20ng_, test_20ng_, inlier_topic, dataset_name, type_tac, anomaly_rate, True)

In [ ]:
if type_emb == 'bert':
    tokenizer = AutoTokenizer.from_pretrained(emb_model)
    vocab = None

elif type_emb in ('glove', 'fasttext'):
    corpus = train_inlier_reuters['text']
    vocab = build_vocab(corpus,min_freq=5)
    tokenizer = None

In [ ]:
cvdd_model, dl_train, dl_test = cvdd_model_pipeline(train_inlier_reuters, test_reuters, attention_size, n_attention_heads, 
                                               type_emb, 100, 64, True, device, tokenizer, vocab)

In [ ]:
cvdd_trainer = cvdd_Net.CVDDTrainer(optimizer_name='adam', learning_rate=lr, lr_milestones=lr_milestones,
                                    n_epochs=n_epochs, lambda_p=lambda_p,
                                    alpha_scheduler=alpha_scheduler, weight_decay=1e-4, device=device)

model_trained = cvdd_trainer.train(cvdd_model, dl_train)

In [ ]:
auc, ap, fpr95, _ = cvdd_trainer.test(model_trained, dl_test, ad_score='context_dist_mean')
print(f"AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")

In [ ]:
type_emb = 'glove'
# emb_model = 'distilbert-base-uncased'
attention_size = 150
n_attention_heads = 10
lr = 0.01
lr_milestones = (40, 60)
n_epochs = 100
lambda_p = 1.0
alpha_scheduler = 'logarithmic'


dataset_name = '20newsgroups'
inlier_topics = ['computer', 'recreation', 'science', 'miscellaneous', 'politics', 'religion']
type_tac = 'ruff'
anomaly_rate = 0.1
# type_tac = 'fate'
# inlier_topics = ['World', 'Sports', 'Business', 'Sci-Tech']
# inlier_topics = ['earn', 'trade', 'acq', 'money-fx', 'crude', 'ship', 'interest']
# inlier_topics = ['money-fx', 'interest']

for inlier_topic in inlier_topics:
    
    list_auc_cvdd = []
    list_fpr_cvdd = []
    list_ap_cvdd = []
    list_time_cvdd = []
        
    print("---------------------------------")
    print(f"-----------{inlier_topic}------------")
    print("---------------------------------\n")
    
    for n_run in range(1,6):
        
        print("///////////////////////////////")
        print(f"////////Run {n_run}////////")
        print("///////////////////////////////\n")
        
        
        train_inlier_reuters, train_anomaly_reuters, val_reuters, test_reuters = train_test_val_split(train_20ng_, test_20ng_, inlier_topic, dataset_name, type_tac, anomaly_rate, False)
        
        corpus = train_inlier_reuters['text']
        vocab = build_vocab(corpus,min_freq=5)
        tokenizer = None

        cvdd_model, dl_train, dl_test = cvdd_model_pipeline(train_inlier_reuters, test_reuters, attention_size, n_attention_heads, 
                                                   type_emb, 500, 64, True, device, tokenizer, vocab)

        cvdd_trainer = cvdd_Net.CVDDTrainer(optimizer_name='adam', learning_rate=lr, lr_milestones=lr_milestones,
                                        n_epochs=n_epochs, lambda_p=lambda_p,
                                        alpha_scheduler=alpha_scheduler, weight_decay=0, device=device)

        model_trained = cvdd_trainer.train(cvdd_model, dl_train)
        
        auc_cvdd, ap_cvdd, fpr95_cvdd, _ = cvdd_trainer.test(model_trained, dl_test, ad_score='context_dist_mean')
        print(f"CVDD --> AUC: {auc_cvdd:.4f} | FPR@95: {fpr95_cvdd:.4f} | AP: {ap_cvdd:.4f}")
        
        list_auc_cvdd.append(auc_cvdd)
        list_fpr_cvdd.append(fpr95_cvdd)    
        list_ap_cvdd.append(ap_cvdd)  
        
        
    print(np.mean(list_auc_cvdd))
    save_results(
    dataset_name=dataset_name, inlier_topic=inlier_topic ,type_emb=type_emb ,ad_model="CVDD",
    auc_mean=np.mean(list_auc_cvdd), ap_mean=np.mean(list_ap_cvdd),fpr_mean=np.mean(list_fpr_cvdd),
    auc_std = np.std(list_auc_cvdd),ap_std =  np.std(list_ap_cvdd),fpr_std = np.std(list_fpr_cvdd),overwrite='naive'
    )


In [ ]:
create_tables()

### RSRAE

In [ ]:
def fit_rsrae(X_inlier, learning_rate=0.001, epoch_size=100):
    
    input_shape = (X_inlier.shape[1],)
    dim_latent = 10
    activation = tf.nn.relu
    loss = 'MSE'
    if_rsr = True
    if_enforce_proj = False
    if_all_alt = False
    renormalize = False
    bn = True
    
    cae = CAE(input_shape=input_shape, hidden_layer_sizes=(16,32,64), intrinsic_size=dim_latent,
                          activation=activation,
                          norm_type='L21', loss_norm_type=loss,
                          if_rsr=if_rsr, enforce_proj=if_enforce_proj, all_alt=if_all_alt,
                          learning_rate=learning_rate,
                          lambda1=0.1, lambda2=0.1,
                          epoch_size=epoch_size, batch_show=epoch_size//2, 
                          normalize=renormalize,
                          bn=bn,
                          random_seed=None)
    
    cae.fit(X_inlier.cpu().numpy(), X_inlier.cpu().numpy())    
    
    return cae

def test_rsrae(X_test, y_test, cae):
    
    with torch.no_grad():
    
        features = cae.get_output(X_test.cpu().numpy())
        flat_output = np.reshape(features, (np.shape(X_test.cpu().numpy())[0], -1))
        flat_input = np.reshape(X_test.cpu().numpy(), (np.shape(X_test.cpu().numpy())[0], -1))

        cosine_similarity = np.sum(flat_output * flat_input, -1) / (np.linalg.norm(flat_output, axis=-1) + 0.000001) / (np.linalg.norm(flat_input, axis=-1) + 0.000001)

        auc = roc_auc_score(y_test, -cosine_similarity)
        ap = average_precision_score(y_test, -cosine_similarity)
        fpr, tpr, thresholds = roc_curve(y_test, -cosine_similarity)
        idx = np.where(tpr >= 0.95)[0][0]
        fpr95 = fpr[idx]

        print(f"RSRAE --> AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")  
    
    return auc, fpr95, ap

In [ ]:
cae = fit_rsrae(X_inlier)
auc, fpr95, ap = test_rsrae(X_test, y_test, cae)

### AE

In [ ]:
def fit_ae(X_inlier):

    clf = AutoEncoder(
        contamination=0.1,
        hidden_neuron_list=[32, 16],  
        hidden_activation_name='relu',
        epoch_num=5,
        batch_size=64,
        dropout_rate=0.0,
        verbose=0
    )

    clf.fit(X_inlier.cpu())
    
    return clf

def test_ae(X_test, y_test, clf):
    scores = clf.decision_function(X_test.cpu())

    auc = roc_auc_score(y_test, scores)
    ap = average_precision_score(y_test, scores)
    fpr, tpr, thresholds = roc_curve(y_test, scores)
    idx = np.where(tpr >= 0.95)[0][0]
    fpr95 = fpr[idx]

    print(f"AE --> AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")  
    
    return auc, fpr95, ap

In [ ]:
clf = fit_ae(X_inlier)
test_ae(X_test, y_test, clf)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


# ============================
# 1) AutoEncoder Model
# ============================

class AutoEncoder(nn.Module):
    def __init__(self, input_dim, hidden_neuron_list=[32, 16], hidden_activation_name='relu'):
        super().__init__()

        # --- Activation ---
        if hidden_activation_name == "relu":
            act = nn.ReLU()
        elif hidden_activation_name == "tanh":
            act = nn.Tanh()
        else:
            raise ValueError("Activation not supported")

        # --- Encoder ---
        encoder_layers = []
        prev_dim = input_dim
        for h in hidden_neuron_list:
            encoder_layers.append(nn.Linear(prev_dim, h))
            encoder_layers.append(act)
            prev_dim = h

        # --- Decoder ---
        decoder_layers = []
        for h in reversed(hidden_neuron_list[:-1]):
            decoder_layers.append(nn.Linear(prev_dim, h))
            decoder_layers.append(act)
            prev_dim = h

        decoder_layers.append(nn.Linear(prev_dim, input_dim))  # final reconstruction

        self.encoder = nn.Sequential(*encoder_layers)
        self.decoder = nn.Sequential(*decoder_layers)

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat


# ============================
# 2) Train Function 
# ============================

def train_autoencoder(model, train_loader, lr=1e-3, n_epochs=20, device="cpu"):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    model.train()
    for epoch in range(n_epochs):
        epoch_loss = 0
        for x_batch in train_loader:
            x_batch = x_batch[0].to(device)

            optimizer.zero_grad()
            x_hat = model(x_batch)

            loss = criterion(x_hat, x_batch)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        print(f"Epoch {epoch+1}/{n_epochs} | Loss = {epoch_loss / len(train_loader):.6f}")

    return model


# ============================
# 3) Test Function (scores)
# ============================

def test_autoencoder(model, test_loader, y_test, device="cpu"):
    model.eval()
    scores = []

    with torch.no_grad():
        for x_batch in test_loader:
            x_batch = x_batch[0].to(device)
            x_hat = model(x_batch)

            # Reconstruction error comme score d'anomalie
            mse = torch.mean((x_batch - x_hat) ** 2, dim=1)
            scores.extend(mse.cpu().numpy())

    auc = roc_auc_score(y_test, scores)
    ap = average_precision_score(y_test, scores)
    fpr, tpr, thresholds = roc_curve(y_test, scores)
    idx = np.where(tpr >= 0.95)[0][0]
    fpr95 = fpr[idx]

    print(f"AE --> AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")          
    
    return scores

In [ ]:
train_loader = DataLoader(TensorDataset(X_inlier), batch_size=32, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test), batch_size=32, shuffle=False)

model = AutoEncoder(
    input_dim=768,
    hidden_neuron_list=[32, 16],
    hidden_activation_name="relu"
)

# Train
model = train_autoencoder(model, train_loader, lr=1e-3, n_epochs=30)

# Test (get anomaly scores)
scores = test_autoencoder(model, test_loader, y_test)

### TCCM

In [ ]:
def fit_tccm(X_inlier):
    n_features = X_inlier.shape[1]

    tccm = TCCM(n_features, epochs=50, learning_rate=0.001, batch_size=64)
    tccm.fit(X_inlier.cpu())
    
    return tccm

def test_tccm(X_test, y_test, tccm):
    scores = tccm.decision_function(X_test)
    
    auc = roc_auc_score(y_test, scores)
    ap = average_precision_score(y_test, scores)
    fpr, tpr, thresholds = roc_curve(y_test, scores)
    idx = np.where(tpr >= 0.95)[0][0]
    fpr95 = fpr[idx]

    print(f"TCCM --> AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")  

    return auc, fpr95, ap

In [ ]:
tccm = fit_tccm(X_inlier)
auc, fpr95, ap = test_tccm(X_test, y_test, tccm)